# Notebook 86 - Paired-Truth p(annotation) x p_print Experiment

Existing paired pages are sufficient for a clean-truth development experiment after regenerating frozen YOLO confidence maps. Because the paired masks did not pass through the production HITL review workflow and differ morphologically from HITL masks, any successful operating region still requires confirmation on new paired clean/annotated pages processed through HITL.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ultralytics pyarrow


Mounted at /content/drive


In [7]:
from pathlib import Path
from datetime import datetime,timezone
import json,hashlib,gc,shutil,importlib.util,sys,cv2,torch
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from ultralytics import YOLO
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject');
MODEL=ROOT/'training_runs'/'20261006_150912_yolo11n-seg'/'weights'/'best.pt';
CLEAN=ROOT/'source'/'clean';ANN=ROOT/'aligned';
MASK=ROOT/'masks'/'corrected';
MAN=ROOT/'development_score_manifest.csv';
OP=ROOT/'v8_9_release'/'v8_9_operator.py';
PB=ROOT/'observed_print_full_development_freeze_v8_2f'/'20261117_120424_v8_2f_full_development_p_print_freeze'/'p_print_provider'
RB=ROOT/'m5_remove_full_development_freeze_v8_8f'/'20261117_110512_notebook52f_v8_8_full_development_remove_freeze'/'full_development_remove'
RUN=ROOT/'notebook86_annotation_print_truth'/datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_notebook86')
for s in ['confidence','pages','tables','visuals','upload']:(RUN/s).mkdir(parents=True,exist_ok=True)
AT=[.925,.900,.875,.850,.825,.800,.775,.750];PC=[.1,.2,.3,.4,.5];DARK=200;MARGIN=5;NBOOT=10000;EXT={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def req(v,msg):
 if not v: raise RuntimeError(msg)
def gray(p):
 x=cv2.imread(str(p),0);req(x is not None,f'Cannot read {p}');return x
def mp(d): return {p.stem:p for p in Path(d).iterdir() if p.is_file() and p.suffix.lower() in EXT}
def sh(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
req(torch.cuda.is_available(),'Select T4 GPU');spec=importlib.util.spec_from_file_location('v89n86',OP);mod=importlib.util.module_from_spec(spec);sys.modules[spec.name]=mod;spec.loader.exec_module(mod);op=mod.load_v89_operator(p_print_bundle_root=PB,remove_bundle_root=RB)
clean,ann,mask=mp(CLEAN),mp(ANN),mp(MASK);man=pd.read_csv(MAN,dtype=str).fillna('');meta=man.set_index('page');pages=sorted(set(clean)&set(ann)&set(mask)&set(man.page));rows=[]
for p in pages:
 o,k,s=gray(ann[p]),gray(clean[p]),gray(mask[p]);req(o.shape==k.shape==s.shape,f'{p}: shape mismatch');rows.append({'page':p,'source_group':meta.loc[p,'source_group'],'annotated_path':str(ann[p]),'clean_path':str(clean[p]),'seed_path':str(mask[p]),'seed_pixels':int((s>0).sum())})
roster=pd.DataFrame(rows);roster.to_csv(RUN/'tables'/'paired_roster.csv',index=False);print(len(roster),'paired pages')


82 paired pages


In [8]:
Y=YOLO(str(MODEL));TS=640;ST=320
def pos(n):
 if n<=TS:return [0]
 a=list(range(0,n-TS+1,ST))
 if a[-1]!=n-TS:a.append(n-TS)
 return a
def confmap(g):
 h,w=g.shape;ph,pw=max(h,TS),max(w,TS);can=np.full((ph,pw,3),255,np.uint8);can[:h,:w]=cv2.cvtColor(g,cv2.COLOR_GRAY2BGR);q=np.zeros((ph,pw),np.float32)
 for y in pos(ph):
  for x in pos(pw):
   r=Y.predict(can[y:y+TS,x:x+TS],conf=.20,imgsz=640,device=0,verbose=False,retina_masks=True)[0]
   if r.masks is None:continue
   for a,b in zip(r.masks.data.cpu().numpy(),r.boxes.conf.cpu().numpy()):
    a=cv2.resize(a,(TS,TS),interpolation=cv2.INTER_NEAREST)>.5;z=q[y:y+TS,x:x+TS];z[a&(b>z)]=b
 return q[:h,:w]
for _,r in roster.iterrows():
 f=RUN/'confidence'/f'{r.page}.npy'
 if not f.exists(): np.save(f,confmap(gray(r.annotated_path)).astype(np.float16));print(r.page)


Page002
Page003
Page004
Page005
Page006
Page007
Page008
Page009
Page010
Page011
Page012
Page013
Page014
Page015
Page016
Page017
Page018
Page019
Page020
Page021
Page022
Page023
Page024
Page025
Page026
Page027
Page028
Page029
Page030
Page031
Page032
Page033
Page034
Page035
Page036
Page037
Page038
Page039
Page040
Page041
Page042
Page044
Page046
Page047
Page048
Page049
Page051
Page052
Page053
Page054
Page055
Page056
Page057
Page058
Page059
Page060
Page061
Page062
Page063
Page064
Page065
Page066
Page067
Page068
Page069
Page070
Page071
Page072
Page073
Page074
Page075
Page076
Page077
Page078
Page079
Page080
Page081
Page082
Page083
Page084
Page085
Page201


In [10]:
out=[]
for _,r in roster.iterrows():
 p=str(r.page)
 if int(r.seed_pixels)==0:out.append({'page':p,'status':'SKIP_ZERO_AUTHORITY'});continue
 o,s=gray(r.annotated_path),gray(r.seed_path);res=op.run_page(o,s,page=p);authority=mod.prepare_mask(s);telea=mod.telea_reconstruct(o,authority);base,routes=mod.geometry_t80(o,authority,telea);req(np.array_equal(base,res.baseline_v78),f'{p}: baseline mismatch');amb=np.asarray(routes['ambiguous'],bool)
 if not amb.any():out.append({'page':p,'status':'SKIP_ZERO_AMBIGUOUS'});continue
 a1=np.asarray(res.corrected,np.uint8).copy();a1[amb]=o[amb];d=RUN/'pages'/p;d.mkdir(exist_ok=True);cv2.imwrite(str(d/'a1.png'),a1);np.savez_compressed(d/'observable.npz',p_annotation=np.load(RUN/'confidence'/f'{p}.npy').astype(np.float32),p_print=np.asarray(res.p_print,np.float32),paper=mod.paper(o,authority),ambiguous=amb);out.append({'page':p,'status':'PROCESSED'});print(p)
outputs=pd.DataFrame(out);outputs.to_csv(RUN/'tables'/'observable_summary.csv',index=False);lock={'state':'CLOSED_BEFORE_CLEAN_TRUTH','operator_sha256':sh(OP),'detector_sha256':sh(MODEL),'processed_pages':int((outputs.status=='PROCESSED').sum()),'clean_truth_read':False};(RUN/'prediction_lock.json').write_text(json.dumps(lock,indent=2))


AttributeError: module 'v89n86' has no attribute 'paper'

In [ ]:
req(json.loads((RUN/'prediction_lock.json').read_text())['state']=='CLOSED_BEFORE_CLEAN_TRUTH','Lock not closed')
def pink(x):return cv2.adaptiveThreshold(x,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
P=set(outputs.loc[outputs.status=='PROCESSED','page']);R=[]
for _,r in roster.iterrows():
 p=str(r.page)
 if p not in P:continue
 d=RUN/'pages'/p;o,k,a1=gray(r.annotated_path),gray(r.clean_path),gray(d/'a1.png')
 with np.load(d/'observable.npz') as z:pa=z['p_annotation'].astype(float);pp=z['p_print'].astype(float);paper=z['paper'];amb=z['ambiguous'].astype(bool)
 kt=cv2.dilate(pink(k).astype(np.uint8),np.ones((3,3),np.uint8))>0;tp=amb&kt;ta=amb&(o<DARK)&~kt;tu=amb&~tp&~ta
 for at in AT:
  for pc in PC:
   cm=amb&(pa>=at)&np.isfinite(pp)&(pp<pc);R.append({'page':p,'source_group':r.source_group,'p_annotation_min':at,'p_print_max':pc,'candidate_pixels':int(cm.sum()),'truth_print_pixels':int(tp.sum()),'truth_annotation_pixels':int(ta.sum()),'cleanup_truth_annotation':int((cm&ta).sum()),'cleanup_truth_print':int((cm&tp).sum()),'cleanup_truth_uncertain':int((cm&tu).sum())})
scores=pd.DataFrame(R);scores.to_csv(RUN/'tables'/'joint_by_page.csv',index=False);pooled=scores.groupby(['p_annotation_min','p_print_max'],as_index=False).sum(numeric_only=True);pooled['annotation_per_print_lost']=pooled.cleanup_truth_annotation/pooled.cleanup_truth_print.clip(lower=1);pooled.to_csv(RUN/'tables'/'joint_pooled.csv',index=False);display(pooled.sort_values(['cleanup_truth_print','cleanup_truth_annotation'],ascending=[True,False]))


In [ ]:
def boot(v,seed):
 v=np.asarray(v,float);rng=np.random.default_rng(seed);m=np.array([rng.choice(v,len(v),True).mean() for _ in range(NBOOT)]);return v.mean(),np.quantile(m,.025),np.quantile(m,.975)
B=[]
for (at,pc),g in scores.groupby(['p_annotation_min','p_print_max']):
 aa=g.cleanup_truth_annotation/g.truth_annotation_pixels.clip(lower=1);pl=g.cleanup_truth_print/g.truth_print_pixels.clip(lower=1);am,al,ah=boot(aa,42+int(at*1000)+int(pc*100));pm,lo,hi=boot(pl,10042+int(at*1000)+int(pc*100));B.append({'p_annotation_min':at,'p_print_max':pc,'pages':g.page.nunique(),'mean_additional_annotation_cleanup':am,'annotation_cleanup_ci95_low':al,'annotation_cleanup_ci95_high':ah,'mean_genuine_print_loss':pm,'print_loss_ci95_low':lo,'print_loss_ci95_high':hi,'pages_with_print_loss':int((g.cleanup_truth_print>0).sum())})
boot=pd.DataFrame(B);boot.to_csv(RUN/'tables'/'joint_bootstrap.csv',index=False);display(boot.sort_values(['print_loss_ci95_high','mean_additional_annotation_cleanup'],ascending=[True,False]))
fig,ax=plt.subplots(figsize=(11,8))
for _,r in boot.iterrows():ax.scatter(r.mean_genuine_print_loss*100,r.mean_additional_annotation_cleanup*100);ax.annotate(f'{r.p_annotation_min:.3f}/{r.p_print_max:.1f}',(r.mean_genuine_print_loss*100,r.mean_additional_annotation_cleanup*100),fontsize=7)
ax.set(xlabel='Mean genuine-print loss versus A1 (%)',ylabel='Mean additional annotation cleanup versus A1 (%)',title='A1 safe-cleanup frontier');ax.grid(alpha=.3);fig.tight_layout();fig.savefig(RUN/'visuals'/'frontier.png',dpi=180);plt.show()
for val,title,name in [('mean_additional_annotation_cleanup','Additional annotation cleanup','cleanup_heatmap.png'),('mean_genuine_print_loss','Genuine print loss','print_loss_heatmap.png')]:
 z=boot.pivot(index='p_annotation_min',columns='p_print_max',values=val).sort_index(ascending=False);fig,ax=plt.subplots(figsize=(9,7));im=ax.imshow(z.to_numpy()*100,aspect='auto',cmap='turbo');ax.set_xticks(range(len(z.columns)),z.columns);ax.set_yticks(range(len(z.index)),z.index);ax.set(xlabel='Maximum p_print',ylabel='Minimum p(annotation)',title=title+' (%)');fig.colorbar(im,ax=ax);fig.tight_layout();fig.savefig(RUN/'visuals'/name,dpi=180);plt.show()


In [ ]:
safe=boot[boot.print_loss_ci95_high<=.0005].sort_values('mean_additional_annotation_cleanup',ascending=False);verdict={'automatic_threshold_promotion':False,'existing_paired_masks_are_not_hitl_profile':True,'production_hitl_confirmation_required':True,'best_candidates_under_0_05pct_upper_print_loss':safe.head(10).to_dict('records')};(RUN/'notebook86_verdict.json').write_text(json.dumps(verdict,indent=2));(RUN/'executive_summary.txt').write_text('Notebook 86 complete. No threshold is promoted automatically. Production confirmation requires new paired pages processed through HITL.');arts=[RUN/'notebook86_verdict.json',RUN/'executive_summary.txt',RUN/'tables'/'joint_pooled.csv',RUN/'tables'/'joint_bootstrap.csv',RUN/'visuals'/'frontier.png',RUN/'visuals'/'cleanup_heatmap.png',RUN/'visuals'/'print_loss_heatmap.png',RUN/'prediction_lock.json']
for p in arts:shutil.copy2(p,RUN/'upload'/p.name)
shutil.make_archive(str(RUN/'notebook86_bundle'),'zip',RUN/'upload');print(RUN/'notebook86_bundle.zip')
